# RePaint Dataset Preparation and Training


Use this notebook to generate synthetic defects (1 glass + 2 wood per image) and to fine-tune a RePaint diffusion model on the resulting pairs. Run each section in order within your ngrok-backed Jupyter environment.


In [ ]:
%pip install --quiet pillow numpy opencv-python matplotlib pandas tqdm scikit-image diffusers accelerate transformers datasets


In [ ]:
import json
import math
import random
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import cv2


## Configuration

Update the paths below to reflect your dataset layout. Expected directory structure under `RAW_DATASET_ROOT`:

- `backgrounds/`: clean surface or product images
- `defects/glass/`: RGBA patches for glass defects
- `defects/wood/`: RGBA patches for wood defects

The conversion script creates a new folder `converted_repaint/` containing clean targets, defected sources, binary masks, and a metadata CSV that ties them together.


In [ ]:
RAW_DATASET_ROOT = Path('/home/jovyan/work/Ayan/repaint')
BACKGROUND_DIR = RAW_DATASET_ROOT / 'backgrounds'
GLASS_DIR = RAW_DATASET_ROOT / 'defects' / 'glass'
WOOD_DIR = RAW_DATASET_ROOT / 'defects' / 'wood'
CONVERTED_DATASET_ROOT = RAW_DATASET_ROOT / 'converted_repaint'

IMAGE_SIZE: Tuple[int, int] = (512, 512)
VARIATIONS_PER_IMAGE = 3
GLASS_COUNT_PER_IMAGE = 1
WOOD_COUNT_PER_IMAGE = 2
PATCH_SCALE_RANGE = (0.6, 1.2)
PATCH_ROTATION_RANGE = (-45.0, 45.0)
MASK_ALPHA_THRESHOLD = 0.2
WIPE_CONVERTED_OUTPUT = True
RNG_SEED = 2025

np.random.seed(RNG_SEED)
random.seed(RNG_SEED)


In [ ]:
SUPPORTED_IMAGE_EXTENSIONS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def collect_image_files(directory: Path) -> List[Path]:
    if directory is None or not directory.exists():
        return []
    files: List[Path] = []
    for ext in SUPPORTED_IMAGE_EXTENSIONS:
        files.extend(sorted(directory.rglob(f'*{ext}')))
    return files

background_files = collect_image_files(BACKGROUND_DIR)
glass_files = collect_image_files(GLASS_DIR)
wood_files = collect_image_files(WOOD_DIR)

print(f'Found {len(background_files)} backgrounds, {len(glass_files)} glass patches, {len(wood_files)} wood patches.')
if len(background_files) == 0:
    raise FileNotFoundError(f'No background images found under {BACKGROUND_DIR}. Please update `BACKGROUND_DIR`.')
if len(glass_files) == 0:
    raise FileNotFoundError(f'No glass patch images found under {GLASS_DIR}. Provide at least one RGBA asset with transparency.')
if len(wood_files) == 0:
    raise FileNotFoundError(f'No wood patch images found under {WOOD_DIR}. Provide at least one RGBA asset with transparency.')


In [ ]:
@dataclass
class PatchAsset:
    path: Path
    image: Image.Image
    alpha: Image.Image
    label_name: str
    label_id: int

def load_patch_assets(file_paths: List[Path], label_name: str, label_id: int) -> List[PatchAsset]:
    assets: List[PatchAsset] = []
    for path in file_paths:
        rgba = Image.open(path).convert('RGBA')
        alpha = rgba.getchannel('A')
        if alpha.getextrema()[1] == 0:
            raise ValueError(f'Patch at {path} has no visible alpha mask. Ensure defects include transparency.')
        assets.append(
            PatchAsset(
                path=path,
                image=rgba.convert('RGB'),
                alpha=alpha,
                label_name=label_name,
                label_id=label_id,
            )
        )
    return assets

glass_assets = load_patch_assets(glass_files, 'glass', 1)
wood_assets = load_patch_assets(wood_files, 'wood', 2)
print(f'Loaded {len(glass_assets)} glass assets and {len(wood_assets)} wood assets.')


In [ ]:
def resize_and_rotate(patch_asset: PatchAsset, rng: np.random.Generator):
    scale = float(rng.uniform(*PATCH_SCALE_RANGE))
    angle = float(rng.uniform(*PATCH_ROTATION_RANGE))
    patch = patch_asset.image
    alpha = patch_asset.alpha
    new_w = max(8, int(patch.width * scale))
    new_h = max(8, int(patch.height * scale))
    patch = patch.resize((new_w, new_h), resample=Image.BICUBIC)
    alpha = alpha.resize((new_w, new_h), resample=Image.BILINEAR)
    patch = patch.rotate(angle, resample=Image.BICUBIC, expand=True)
    alpha = alpha.rotate(angle, resample=Image.BILINEAR, expand=True)
    return patch, alpha

def blend_patch(base_array: np.ndarray, label_mask: np.ndarray, patch_img: Image.Image, alpha_img: Image.Image, top_left: Tuple[int, int], label_value: int):
    x0, y0 = top_left
    patch_arr = np.asarray(patch_img, dtype=np.float32)
    alpha_arr = np.asarray(alpha_img, dtype=np.float32) / 255.0
    if patch_arr.ndim == 2:
        patch_arr = np.repeat(patch_arr[..., None], 3, axis=2)
    h, w = patch_arr.shape[:2]
    x1 = min(base_array.shape[1], x0 + w)
    y1 = min(base_array.shape[0], y0 + h)
    if x1 <= x0 or y1 <= y0:
        return base_array, label_mask
    patch_crop = patch_arr[: y1 - y0, : x1 - x0]
    alpha_crop = alpha_arr[: y1 - y0, : x1 - x0][..., None]
    base_region = base_array[y0:y1, x0:x1]
    blended = patch_crop * alpha_crop + base_region * (1.0 - alpha_crop)
    base_array[y0:y1, x0:x1] = blended.astype(np.uint8)
    label_region = label_mask[y0:y1, x0:x1]
    label_mask[y0:y1, x0:x1] = np.where(alpha_crop[..., 0] > MASK_ALPHA_THRESHOLD, label_value, label_region)
    return base_array, label_mask

def apply_defects(clean_image: Image.Image, rng: np.random.Generator):
    pristine_array = np.asarray(clean_image.convert('RGB'))
    corrupted_array = pristine_array.copy()
    label_mask = np.zeros(pristine_array.shape[:2], dtype=np.uint8)
    usage: Dict[str, List[str]] = {'glass': [], 'wood': []}

    def place_from_assets(assets: List[PatchAsset], count: int, label_value: int, usage_key: str):
        nonlocal corrupted_array, label_mask
        if count <= 0:
            return
        indices = rng.choice(len(assets), size=count, replace=len(assets) < count)
        for idx in np.atleast_1d(indices):
            asset = assets[int(idx)]
            patch_img, alpha_img = resize_and_rotate(asset, rng)
            max_x = max(1, corrupted_array.shape[1] - patch_img.width)
            max_y = max(1, corrupted_array.shape[0] - patch_img.height)
            x0 = int(rng.integers(0, max_x + 1))
            y0 = int(rng.integers(0, max_y + 1))
            corrupted_array, label_mask = blend_patch(
                corrupted_array, label_mask, patch_img, alpha_img, (x0, y0), label_value
            )
            usage[usage_key].append(asset.path.name)

    place_from_assets(glass_assets, GLASS_COUNT_PER_IMAGE, 1, 'glass')
    place_from_assets(wood_assets, WOOD_COUNT_PER_IMAGE, 2, 'wood')

    binary_mask = (label_mask > 0).astype(np.uint8) * 255

    clean_target = Image.fromarray(pristine_array, mode='RGB')
    defected_source = Image.fromarray(corrupted_array, mode='RGB')
    binary_mask_img = Image.fromarray(binary_mask, mode='L')
    label_mask_img = Image.fromarray(label_mask, mode='L')

    return clean_target, defected_source, binary_mask_img, label_mask_img, usage


In [ ]:
if WIPE_CONVERTED_OUTPUT and CONVERTED_DATASET_ROOT.exists():
    print(f'Clearing existing converted dataset at {CONVERTED_DATASET_ROOT}')
    shutil.rmtree(CONVERTED_DATASET_ROOT)

clean_dir = CONVERTED_DATASET_ROOT / 'clean'
defected_dir = CONVERTED_DATASET_ROOT / 'defected'
mask_dir = CONVERTED_DATASET_ROOT / 'masks_binary'
label_mask_dir = CONVERTED_DATASET_ROOT / 'masks_labels'
clean_dir.mkdir(parents=True, exist_ok=True)
defected_dir.mkdir(parents=True, exist_ok=True)
mask_dir.mkdir(parents=True, exist_ok=True)
label_mask_dir.mkdir(parents=True, exist_ok=True)

metadata_records: List[Dict[str, str]] = []
global_index = 0
for bg_idx, bg_path in enumerate(tqdm(background_files, desc='Converting backgrounds')):
    base_image = Image.open(bg_path).convert('RGB')
    if IMAGE_SIZE is not None:
        base_image = base_image.resize(IMAGE_SIZE, resample=Image.BICUBIC)
    for variation_idx in range(VARIATIONS_PER_IMAGE):
        rng = np.random.default_rng(RNG_SEED + bg_idx * 997 + variation_idx)
        clean_img, defected_img, mask_img, label_img, usage = apply_defects(base_image, rng)
        sample_id = f"{bg_path.stem}_var{variation_idx:02d}"
        clean_path = clean_dir / f"{sample_id}_clean.png"
        defected_path = defected_dir / f"{sample_id}_defected.png"
        mask_path = mask_dir / f"{sample_id}_mask.png"
        label_path = label_mask_dir / f"{sample_id}_label.png"
        clean_img.save(clean_path)
        defected_img.save(defected_path)
        mask_img.save(mask_path)
        label_img.save(label_path)
        metadata_records.append(
            {
                'sample_id': sample_id,
                'background_source': str(bg_path.relative_to(RAW_DATASET_ROOT)),
                'clean_path': str(clean_path.relative_to(CONVERTED_DATASET_ROOT)),
                'defected_path': str(defected_path.relative_to(CONVERTED_DATASET_ROOT)),
                'mask_path': str(mask_path.relative_to(CONVERTED_DATASET_ROOT)),
                'label_mask_path': str(label_path.relative_to(CONVERTED_DATASET_ROOT)),
                'glass_assets': json.dumps(usage['glass']),
                'wood_assets': json.dumps(usage['wood']),
            }
        )
        global_index += 1

metadata_df = pd.DataFrame(metadata_records)
metadata_path = CONVERTED_DATASET_ROOT / 'metadata.csv'
metadata_df.to_csv(metadata_path, index=False)
print(f'Wrote {len(metadata_df)} samples to {CONVERTED_DATASET_ROOT}')


In [ ]:
if 'metadata_df' not in globals():
    metadata_df = pd.read_csv(CONVERTED_DATASET_ROOT / 'metadata.csv')

sample = metadata_df.sample(1, random_state=RNG_SEED).iloc[0]
clean_preview = Image.open(CONVERTED_DATASET_ROOT / sample['clean_path']).convert('RGB')
defected_preview = Image.open(CONVERTED_DATASET_ROOT / sample['defected_path']).convert('RGB')
mask_preview = Image.open(CONVERTED_DATASET_ROOT / sample['mask_path']).convert('L')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(clean_preview)
axes[0].set_title('Clean target')
axes[0].axis('off')
axes[1].imshow(defected_preview)
axes[1].set_title('Synthetic defects (input)')
axes[1].axis('off')
axes[2].imshow(mask_preview, cmap='gray')
axes[2].set_title('Binary mask (1 glass + 2 wood)')
axes[2].axis('off')
plt.tight_layout()
plt.show()


## Training a RePaint Diffusion Model

The following section sets up a pixel-space diffusion model trained with conditional inputs (defected image + binary mask). We train with a DDPM objective and later reuse the weights with the RePaint scheduler for inpainting. Adjust hyperparameters to match your available GPU memory and dataset size.


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from accelerate import Accelerator
from diffusers import UNet2DModel, DDPMScheduler, RePaintScheduler, RePaintPipeline
from diffusers.optimization import get_scheduler
from diffusers.utils import make_image_grid


In [ ]:
class RepaintDefectDataset(Dataset):
    def __init__(self, metadata: pd.DataFrame, root_dir: Path, image_size: Tuple[int, int]):
        self.metadata = metadata.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.image_transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=InterpolationMode.NEAREST),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
        ])

    def __len__(self) -> int:
        return len(self.metadata)

    def __getitem__(self, idx: int):
        row = self.metadata.iloc[idx]
        clean = Image.open(self.root_dir / row['clean_path']).convert('RGB')
        defected = Image.open(self.root_dir / row['defected_path']).convert('RGB')
        mask = Image.open(self.root_dir / row['mask_path']).convert('L')

        clean_tensor = self.image_transform(clean) * 2.0 - 1.0
        defected_tensor = self.image_transform(defected) * 2.0 - 1.0
        mask_tensor = self.mask_transform(mask)

        return {
            'clean': clean_tensor,
            'defected': defected_tensor,
            'mask': mask_tensor,
        }


In [ ]:
TRAIN_BATCH_SIZE = 4
NUM_EPOCHS = 50
LEARNING_RATE = 5e-5
GRADIENT_ACCUMULATION_STEPS = 1
LR_SCHEDULER_TYPE = 'cosine'
LR_WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0
NUM_TRAIN_TIMESTEPS = 1000
MIXED_PRECISION = 'fp16'
NUM_WORKERS = 4
CHECKPOINT_INTERVAL = 5
CHECKPOINT_DIR = CONVERTED_DATASET_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

training_metadata = pd.read_csv(CONVERTED_DATASET_ROOT / 'metadata.csv')
dataset = RepaintDefectDataset(training_metadata, CONVERTED_DATASET_ROOT, IMAGE_SIZE)
train_dataloader = DataLoader(
    dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
total_training_steps = NUM_EPOCHS * math.ceil(len(dataset) / TRAIN_BATCH_SIZE)
print(f'Training on {len(dataset)} samples ({total_training_steps} gradient steps)')


In [ ]:
unet = UNet2DModel(
    sample_size=IMAGE_SIZE[0],
    in_channels=7,  # noisy clean (3) + defected context (3) + mask (1)
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(128, 256, 256, 512),
    down_block_types=(
        'DownBlock2D',
        'DownBlock2D',
        'DownBlock2D',
        'AttnDownBlock2D',
    ),
    up_block_types=(
        'AttnUpBlock2D',
        'UpBlock2D',
        'UpBlock2D',
        'UpBlock2D',
    ),
)
noise_scheduler = DDPMScheduler(
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    beta_schedule='squaredcos_cap_v2',
)
optimizer = torch.optim.AdamW(unet.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
lr_scheduler = get_scheduler(
    name=LR_SCHEDULER_TYPE,
    optimizer=optimizer,
    num_warmup_steps=LR_WARMUP_STEPS,
    num_training_steps=total_training_steps,
)
accelerator = Accelerator(
    mixed_precision=MIXED_PRECISION,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)
unet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
    unet, optimizer, train_dataloader, lr_scheduler
)
print(f'Using device: {accelerator.device}')


In [ ]:
global_step = 0
for epoch in range(NUM_EPOCHS):
    unet.train()
    progress_bar = tqdm(total=len(train_dataloader), disable=not accelerator.is_local_main_process)
    progress_bar.set_description(f'Epoch {epoch + 1}/{NUM_EPOCHS}')
    for step, batch in enumerate(train_dataloader):
        with accelerator.accumulate(unet):
            clean = batch['clean']
            defected = batch['defected']
            mask = batch['mask']

            noise = torch.randn_like(clean)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (clean.shape[0],),
                device=clean.device, dtype=torch.long
            )
            noisy_clean = noise_scheduler.add_noise(clean, noise, timesteps)
            model_input = torch.cat([noisy_clean, defected, mask], dim=1)
            noise_pred = unet(model_input, timesteps).sample
            loss = F.mse_loss(noise_pred, noise)

            accelerator.backward(loss)
            accelerator.clip_grad_norm_(unet.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        progress_bar.update(1)
        progress_bar.set_postfix(loss=loss.item())
        global_step += 1
    progress_bar.close()

    if accelerator.is_main_process and ((epoch + 1) % CHECKPOINT_INTERVAL == 0 or (epoch + 1) == NUM_EPOCHS):
        ckpt_path = CHECKPOINT_DIR / f'unet_epoch_{epoch + 1:04d}.pt'
        torch.save(accelerator.unwrap_model(unet).state_dict(), ckpt_path)
        print(f'Saved checkpoint to {ckpt_path}')


In [ ]:
def tensor_to_pil(image_tensor: torch.Tensor) -> Image.Image:
    array = image_tensor.detach().cpu().clamp(-1, 1)
    array = (array + 1.0) / 2.0
    array = (array * 255).to(torch.uint8)
    array = array.permute(1, 2, 0).numpy()
    return Image.fromarray(array)

def pil_to_tensor(image: Image.Image) -> torch.Tensor:
    tensor = transforms.ToTensor()(image) * 2.0 - 1.0
    return tensor


In [ ]:
# Example inference using the latest checkpoint
checkpoints = sorted(CHECKPOINT_DIR.glob('unet_epoch_*.pt'))
if not checkpoints:
    raise FileNotFoundError(f'No checkpoints found in {CHECKPOINT_DIR}. Run the training section first.')
latest_ckpt = checkpoints[-1]
print(f'Loading checkpoint {latest_ckpt}')
inference_unet = UNet2DModel(
    sample_size=IMAGE_SIZE[0],
    in_channels=7,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(128, 256, 256, 512),
    down_block_types=('DownBlock2D', 'DownBlock2D', 'DownBlock2D', 'AttnDownBlock2D'),
    up_block_types=('AttnUpBlock2D', 'UpBlock2D', 'UpBlock2D', 'UpBlock2D'),
)
inference_unet.load_state_dict(torch.load(latest_ckpt, map_location='cpu'))
inference_unet.eval()
device = accelerator.device if 'accelerator' in globals() else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
inference_unet.to(device)

repaint_scheduler = RePaintScheduler(
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    beta_schedule='squaredcos_cap_v2',
    clip_sample=True,
)
repaint_pipeline = RePaintPipeline(unet=inference_unet, scheduler=repaint_scheduler)
repaint_pipeline = repaint_pipeline.to(device)

test_row = training_metadata.sample(1, random_state=RNG_SEED + 42).iloc[0]
defected_pil = Image.open(CONVERTED_DATASET_ROOT / test_row['defected_path']).convert('RGB')
mask_pil = Image.open(CONVERTED_DATASET_ROOT / test_row['mask_path']).convert('L')

generator = torch.Generator(device=device).manual_seed(RNG_SEED)
result = repaint_pipeline(
    image=defected_pil,
    mask_image=mask_pil,
    num_inference_steps=250,
    eta=1.0,
    jump_length=10,
    jump_n_sample=10,
    generator=generator,
)
grid = make_image_grid([defected_pil, mask_pil.convert('RGB'), result.images[0]], rows=1, cols=3)
display(grid)
